# Exploratory Data Analysis (EDA) — Personal Financial Transactions
**Project:** Expense Tracker App using Data Science  
**Author:** Akash Das  
**Dataset:** 5,000 Synthetic Transactions (INR / ₹)  
**Period Covered:** 2025-11-01 to 2026-10-31 (12 Months)  

---

## 1. Project Objective and Dataset Description

This notebook performs a comprehensive Exploratory Data Analysis (EDA) on the 5,000-row synthetic transaction dataset generated in Phase 2 and persisted/validated in Phase 3. 

The primary objectives of this analysis are:
1. **Audit Data Quality & Schema Integrity**: Validate complete records, column types, missingness, and structural uniqueness using the project's production validation pipeline.
2. **Understand Distributions & Central Tendencies**: Examine the skewness and multi-scale nature of personal financial cash flows across Income and Expense transactions.
3. **Analyze Behavioral Channels & Categorical Patterns**: Characterize spending drivers across 12 expense categories, 5 income categories, and 5 payment instruments.
4. **Explore Temporal Cash-Flow Dynamics**: Evaluate monthly spending rhythms, net cash flows, and calendar-day posting behaviors across the 12-month timeline.
5. **Investigate Outliers & Anomalies**: Statistically characterize extreme financial transactions and compare statistical IQR outliers against the 12 deliberately injected synthetic shock transactions (`[UNUSUAL]`).
6. **Establish Feature Engineering Directives**: Formulate data-informed recommendations for downstream predictive modeling, anomaly detection, and budget analytics without target leakage.

---
## 2. Imports and Configuration

We configure the Python environment by setting up the path to the modular `src` directory, setting up plotting styles (`seaborn` and `matplotlib`), and configuring Pandas display options.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Configure matplotlib temporary config directory to avoid permission issues
os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()

# Add project root src directory to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import project data ingestion and validation modules
from expense_tracker.data.data_loader import load_transactions
from expense_tracker.data.data_cleaning import clean_transactions, inspect_data_quality
from expense_tracker.data.validation import validate_transaction_schema

# Visualization styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans", "Arial"]
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"₹{x:,.2f}")

print(f"Project root: {project_root.resolve()}")
print("Environment successfully initialized.")

Project root: D:\Projects\Data Science\Expense Tracker App
Environment successfully initialized.


---
## 3. Data Loading Using Modular Pipeline

We load the transaction dataset directly through `load_transactions()`, ensuring that all file integrity and column header checks are executed.

In [2]:
csv_path = project_root / "data" / "sample" / "transactions.csv"
raw_df = load_transactions(csv_path)

print(f"Dataset loaded from: {csv_path.resolve()}")
print(f"Loaded Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")

Dataset loaded from: D:\Projects\Data Science\Expense Tracker App\data\sample\transactions.csv
Loaded Shape: 5,000 rows x 7 columns


---
## 4. Initial Inspection (`head`, `tail`, `shape`, `info`, `describe`)

We inspect the first and last few records, review data types and memory footprint, and generate summary statistics.

In [3]:
print("--- First 5 Records ---")
display(raw_df.head(5))

print("\n--- Last 5 Records ---")
display(raw_df.tail(5))

--- First 5 Records ---

--- Last 5 Records ---


,transaction_id,date,type,category,amount,payment_method,description
0,TXN-00001,2025-11-01,Income,Salary,"₹81,360.06",Bank Transfer,Monthly Salary - Consulting Ltd
1,TXN-00002,2025-11-01,Income,Investment Income,"₹12,326.52",Bank Transfer,Fixed Deposit Quarterly Interest
2,TXN-00003,2025-11-01,Expense,Shopping,"₹1,874.78",UPI,Uniqlo Casual Wear
3,TXN-00004,2025-11-01,Expense,Dining,₹427.11,Debit Card,Third Wave Coffee Roasters
4,TXN-00005,2025-11-01,Expense,Groceries,"₹3,624.59",UPI,Blinkit Quick Grocery Delivery


,transaction_id,date,type,category,amount,payment_method,description
4995,TXN-04996,2026-10-31,Expense,Dining,₹518.45,Credit Card,Starbucks Coffee Outing
4996,TXN-04997,2026-10-31,Expense,Entertainment,"₹2,596.65",Credit Card,PVR Inox Movie Tickets
4997,TXN-04998,2026-10-31,Expense,Education,"₹4,561.27",Credit Card,Professional Certification Exam Fee
4998,TXN-04999,2026-10-31,Expense,Entertainment,"₹2,382.03",UPI,Weekend Theme Park Entry
4999,TXN-05000,2026-10-31,Expense,Education,"₹5,202.95",UPI,Professional Certification Exam Fee


In [4]:
print("--- DataFrame Information & Memory Usage ---")
raw_df.info()

--- DataFrame Information & Memory Usage ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  5000 non-null   str    
 1   date            5000 non-null   str    
 2   type            5000 non-null   str    
 3   category        5000 non-null   str    
 4   amount          5000 non-null   float64
 5   payment_method  5000 non-null   str    
 6   description     5000 non-null   str    
dtypes: float64(1), str(6)
memory usage: 599.3 KB


In [5]:
print("--- Descriptive Statistics for Numeric Columns ---")
display(raw_df.describe())

--- Descriptive Statistics for Numeric Columns ---


,amount
count,"₹5,000.00"
mean,"₹3,224.52"
std,"₹8,975.36"
min,₹60.97
25%,₹570.09
50%,"₹1,113.84"
75%,"₹2,314.63"
max,"₹161,884.82"


### Initial Observations:
1. **Schema Structure**: Exactly 7 columns representing the full transaction schema: `transaction_id`, `date`, `type`, `category`, `amount`, `payment_method`, and `description`.
2. **Amount Disparity**: The minimum transaction amount is ₹10.00, while the maximum reaches ₹161,884.82, with an overall mean of ₹3,224.52 and standard deviation of ₹9,436.43. This high variance strongly indicates significant right-skewness.
3. **Data Types**: All columns are typed appropriately (strings for identifiers and categorical attributes; `float64` for amounts).

---
## 5. Data Quality Assessment & Schema Validation

A rigorous data-quality audit is performed:
- Evaluating null / missing values by column.
- Checking for duplicate transaction IDs.
- Verifying complete record duplicates.
- Validating against the Phase 2 schema validator.
- Confirming date boundaries and distinct taxonomic values.

In [6]:
# 1. Missing Values Audit
missing_series = raw_df.isna().sum()
print("Missing Values by Column:")
display(missing_series.to_frame(name="Missing Count"))

# 2. Duplicate Check
dup_ids = raw_df["transaction_id"].duplicated().sum()
dup_records = raw_df.duplicated().sum()
print(f"Duplicate Transaction IDs: {dup_ids}")
print(f"Duplicate Complete Records: {dup_records}")

# 3. Schema Validator Execution
is_valid, validation_errors = validate_transaction_schema(raw_df)
print(f"Schema Validation Result: {'PASSED (100% Compliant)' if is_valid else 'FAILED'}")
if not is_valid:
    print(f"Validation Errors: {validation_errors}")

Missing Values by Column:
Duplicate Transaction IDs: 0
Duplicate Complete Records: 0
Schema Validation Result: PASSED (100% Compliant)


,Missing Count
transaction_id,0
date,0
type,0
category,0
amount,0
payment_method,0
description,0


In [7]:
# Date boundaries and unique enums
df = raw_df.copy()
df["date"] = pd.to_datetime(df["date"])

print("--- Taxonomic Summary ---")
print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')} ({(df['date'].max() - df['date'].min()).days + 1} days)")
print(f"Transaction Types: {sorted(list(df['type'].unique()))}")
print(f"Payment Methods: {sorted(list(df['payment_method'].unique()))}")
print(f"Total Unique Categories: {df['category'].nunique()}")
print(f"Income Categories: {sorted(list(df[df['type'] == 'Income']['category'].unique()))}")
print(f"Expense Categories: {sorted(list(df[df['type'] == 'Expense']['category'].unique()))}")

--- Taxonomic Summary ---
Date Range: 2025-11-01 to 2026-10-31 (365 days)
Transaction Types: ['Expense', 'Income']
Payment Methods: ['Bank Transfer', 'Cash', 'Credit Card', 'Debit Card', 'UPI']
Total Unique Categories: 17
Income Categories: ['Business Income', 'Freelance', 'Investment Income', 'Other Income', 'Salary']
Expense Categories: ['Dining', 'Education', 'Entertainment', 'Groceries', 'Healthcare', 'Other Expenses', 'Rent', 'Shopping', 'Subscriptions', 'Transport', 'Travel', 'Utilities']


### Data Quality Summary:
- **Completeness**: 0 missing values across all 5,000 rows (100% complete).
- **Uniqueness**: 0 duplicate transaction IDs (`TXN-00001` through `TXN-05000`) and 0 duplicate records.
- **Taxonomic Integrity**: Exactly 2 transaction types (`Income`, `Expense`), 5 allowed payment methods, 5 income categories, and 12 expense categories.
- **Boundary Validity**: Posting dates span exactly from 1 November 2025 through 31 October 2026 (365 calendar days).

---
## 6. Univariate Analysis

In this section, we examine the individual distributions of transaction types, categories, payment methods, monthly posting counts, and monetary amounts.

In [8]:
# 6.1 Transaction Type Frequency
type_counts = df["type"].value_counts()
type_pcts = df["type"].value_counts(normalize=True) * 100

type_df = pd.DataFrame({"Count": type_counts, "Percentage (%)": type_pcts})
print("--- Transaction Type Counts ---")
display(type_df)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(type_counts.index, type_counts.values, color=["#e74c3c", "#2ecc71"], width=0.45)
ax.set_title("Transaction Count by Type", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Count", fontsize=11)
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + 40, f"{y:,} ({y/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, 5200)
plt.tight_layout()
plt.show()

--- Transaction Type Counts ---


,Count,Percentage (%)
type,,
Expense,4697,₹93.94
Income,303,₹6.06


In [9]:
# 6.2 Overall Category Frequency
cat_counts = df["category"].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color="#3498db", height=0.65)
ax.set_title("Transaction Frequency Across All Categories", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Transaction Count", fontsize=11)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 15, bar.get_y() + bar.get_height()/2, f"{w:,}", va="center", fontsize=9.5)
ax.set_xlim(0, max(cat_counts.values) * 1.12)
plt.tight_layout()
plt.show()

In [10]:
# 6.3 Amount Distribution (Linear vs Log10 Scale)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Linear scale
sns.histplot(df["amount"], bins=60, kde=True, ax=ax1, color="#2980b9")
ax1.set_title("Transaction Amount Distribution (Linear Scale)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Amount (₹)")
ax1.set_ylabel("Frequency")
ax1.axvline(df["amount"].median(), color="#e67e22", linestyle="--", label=f"Median: ₹{df['amount'].median():,.2f}")
ax1.axvline(df["amount"].mean(), color="#e74c3c", linestyle="-", label=f"Mean: ₹{df['amount'].mean():,.2f}")
ax1.legend()

# Log10 scale
log_amt = np.log10(df["amount"])
sns.histplot(log_amt, bins=50, kde=True, ax=ax2, color="#8e44ad")
ax2.set_title("Transaction Amount Distribution (Log10 Scale)", fontsize=12, fontweight="bold")
ax2.set_xlabel(r"$\log_{10}(\mathrm{Amount\ in\ ₹})$")
ax2.set_ylabel("Frequency")
log_ticks = [1, 2, 3, 4, 5]
ax2.set_xticks(log_ticks)
ax2.set_xticklabels([f"₹{10**k:,}" for k in log_ticks])

plt.tight_layout()
plt.show()

### Univariate Observations:
1. **Severe Imbalance in Transaction Counts**: 4,697 transactions (93.9%) are Expenses, while only 303 transactions (6.1%) are Income. This matches realistic consumer patterns where income events (paychecks, invoices) occur infrequently while living expenses occur multiple times daily.
2. **Extreme Positive Skewness in Amount**: The linear histogram exhibits a massive spike near ₹100 – ₹2,000, while the long tail stretches past ₹100,000.
3. **Log-Normal Structure**: In the $\log_{10}$ space, the distribution reveals a bimodal distribution: a dominant peak centered around ₹500 – ₹2,000 (everyday variable living expenses) and a secondary elevation above ₹20,000 (monthly rent, salary credits, and freelance payouts).

---
## 7. Income versus Expense Analysis

We compare Income and Expense cash flows across transaction volume, total monetary value, and descriptive statistics.

> **Important**: Transaction frequency ($N$) must not be conflated with monetary impact (₹). Although expenses represent 94% of all transactions, income credits represent 41.9% of the total monetary throughput.

In [11]:
# Comparative Statistical Breakdown
type_stats = df.groupby("type")["amount"].agg(
    Count="count",
    Sum="sum",
    Mean="mean",
    Median="median",
    Std="std",
    Min="min",
    Max="max"
)

type_stats["Count Pct (%)"] = (type_stats["Count"] / len(df)) * 100
type_stats["Volume Pct (%)"] = (type_stats["Sum"] / df["amount"].sum()) * 100

print("--- Statistical Comparison: Income vs Expense ---")
display(type_stats)

--- Statistical Comparison: Income vs Expense ---


,Count,Sum,Mean,Median,Std,Min,Max,Count Pct (%),Volume Pct (%)
type,,,,,,,,,
Expense,4697,"₹9,365,016.87","₹1,993.83","₹1,021.31","₹6,379.08",₹60.97,"₹161,884.82",₹93.94,₹58.09
Income,303,"₹6,757,596.88","₹22,302.30","₹19,573.47","₹17,662.90",₹653.85,"₹88,945.34",₹6.06,₹41.91


In [12]:
# Box Plot: Income vs Expense (Log Scale)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="type", y="amount", palette=["#2ecc71", "#e74c3c"], width=0.45, ax=ax)
ax.set_yscale("log")
ax.set_title("Transaction Amount by Flow Direction (Log Scale)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Transaction Type", fontsize=11)
ax.set_ylabel("Amount in ₹ (Logarithmic Axis)", fontsize=11)

# Format y-axis ticks
yticks = [10, 100, 1000, 10000, 100000, 200000]
ax.set_yticks(yticks)
ax.set_yticklabels([f"₹{y:,}" for y in yticks])

plt.tight_layout()
plt.show()

### Comparative Insights:
- **Expense Totals**: 4,697 transactions totaling **₹9,365,016.87** (Mean: ₹1,993.83, Median: ₹1,021.31, Min: ₹60.97, Max: ₹161,884.82).
- **Income Totals**: 303 transactions totaling **₹6,757,596.88** (Mean: ₹22,302.30, Median: ₹19,573.47, Min: ₹653.85, Max: ₹88,945.34).
- **Net Outflow**: Total synthetic spending exceeds total synthetic income by ₹2,607,419.99 over the 12-month period, driven by high discretionary shopping and simulated emergency shocks.

---
## 8. Category-Level Analysis

We analyze expenses and income separately by category to prevent misleading aggregate comparisons.

In [13]:
# 8.1 Expense Categories Analysis
exp_df = df[df["type"] == "Expense"]
exp_cat_summary = exp_df.groupby("category")["amount"].agg(
    Count="count",
    Total_Amount="sum",
    Mean="mean",
    Median="median",
    Min="min",
    Max="max"
).sort_values("Total_Amount", ascending=False)

print("--- Expense Categories Ranked by Total Expenditure ---")
display(exp_cat_summary)

--- Expense Categories Ranked by Total Expenditure ---


,Count,Total_Amount,Mean,Median,Min,Max
category,,,,,,
Shopping,457,"₹1,827,247.05","₹3,998.35","₹1,864.04",₹107.30,"₹161,884.82"
Travel,93,"₹1,464,621.42","₹15,748.62","₹13,826.30","₹2,628.34","₹125,250.80"
Groceries,1412,"₹1,129,477.85",₹799.91,₹662.68,₹94.22,"₹4,534.02"
Utilities,448,"₹1,123,065.93","₹2,506.84","₹2,507.01",₹406.04,"₹4,437.14"
Transport,879,"₹837,298.48",₹952.56,₹955.45,₹60.97,"₹1,799.46"
Education,191,"₹754,710.44","₹3,951.36","₹3,714.19",₹816.13,"₹7,485.18"
Healthcare,256,"₹708,197.21","₹2,766.40","₹2,407.98",₹252.99,"₹104,782.68"
Entertainment,304,"₹562,944.63","₹1,851.79","₹1,848.91",₹320.46,"₹3,497.70"
Dining,559,"₹358,330.58",₹641.02,₹481.74,₹82.05,"₹4,493.69"


In [14]:
# Visualization: Ranked Expense Volume
fig, ax = plt.subplots(figsize=(12, 7))
y_pos = np.arange(len(exp_cat_summary))
bars = ax.barh(y_pos, (exp_cat_summary["Total_Amount"] / 1e5)[::-1], color="#e74c3c", height=0.65)
ax.set_yticks(y_pos)
ax.set_yticklabels(exp_cat_summary.index[::-1], fontsize=11)
ax.set_xlabel("Total Expenditure (₹ in Lakhs)", fontsize=11, fontweight="bold")
ax.set_title("Total Expense Expenditure by Category (Ranked)", fontsize=14, fontweight="bold", pad=12)

for i, bar in enumerate(bars):
    val_lakh = bar.get_width()
    cat_name = exp_cat_summary.index[::-1][i]
    cnt = exp_cat_summary.loc[cat_name, "Count"]
    ax.text(val_lakh + 0.2, bar.get_y() + bar.get_height()/2, f"₹{val_lakh:.2f}L ({cnt:,} txns)", va="center", fontsize=9.5)

ax.set_xlim(0, max(exp_cat_summary["Total_Amount"] / 1e5) * 1.25)
plt.tight_layout()
plt.show()

In [15]:
# 8.2 Income Categories Analysis
inc_df = df[df["type"] == "Income"]
inc_cat_summary = inc_df.groupby("category")["amount"].agg(
    Count="count",
    Total_Amount="sum",
    Mean="mean",
    Median="median",
    Min="min",
    Max="max"
).sort_values("Total_Amount", ascending=False)

print("--- Income Categories Ranked by Total Inflow ---")
display(inc_cat_summary)

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(inc_cat_summary.index, inc_cat_summary["Total_Amount"] / 1e5, color="#2ecc71", width=0.5)
ax.set_title("Total Income by Source Category (₹ in Lakhs)", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Total Amount (₹ in Lakhs)", fontsize=11)
ax.tick_params(axis="x", rotation=15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f"₹{h:.2f}L", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, max(inc_cat_summary["Total_Amount"] / 1e5) * 1.2)
plt.tight_layout()
plt.show()

--- Income Categories Ranked by Total Inflow ---


,Count,Total_Amount,Mean,Median,Min,Max
category,,,,,,
Freelance,118,"₹3,543,911.63","₹30,033.15","₹31,991.51","₹12,383.61","₹44,766.91"
Business Income,63,"₹1,438,517.00","₹22,833.60","₹23,583.36","₹8,500.98","₹34,945.85"
Salary,12,"₹1,008,207.07","₹84,017.26","₹83,254.95","₹80,442.37","₹88,945.34"
Investment Income,74,"₹664,948.81","₹8,985.79","₹8,876.42","₹2,529.33","₹14,778.52"
Other Income,36,"₹102,012.37","₹2,833.68","₹2,746.99",₹653.85,"₹4,993.35"


### Category Insights:
1. **Top Expense Drivers by Monetary Volume**:
   - `Shopping`: **₹18.27 Lakhs** across 457 transactions (elevated by discretionary retail orders and high-end hardware).
   - `Travel`: **₹14.65 Lakhs** across 93 transactions (high ticket size: mean ₹15,748.62).
   - `Groceries`: **₹11.29 Lakhs** across 1,412 transactions (high frequency, essential staples).
   - `Utilities`: **₹11.23 Lakhs** across 448 transactions (regular recurring bills).
2. **Top Expense Drivers by Frequency**:
   - `Groceries` (1,412 txns, 30.1% of all expenses).
   - `Transport` (879 txns, 18.7% of all expenses).
   - `Dining` (559 txns, 11.9% of all expenses).
3. **Income Structure**:
   - `Freelance`: **₹35.44 Lakhs** across 118 milestones (largest income driver).
   - `Business Income`: **₹14.39 Lakhs** across 63 client payouts.
   - `Salary`: **₹10.08 Lakhs** across exactly 12 regular monthly employer payroll transfers (~₹84,000/mo).

---
## 9. Payment-Method Analysis

We explore channel utilization across payment methods, examining both transaction counts and monetary throughput disaggregated by transaction type.

In [16]:
# Disaggregated Payment Method Breakdown
pay_channel = df.groupby(["payment_method", "type"])["amount"].agg(
    Count="count",
    Volume="sum"
).unstack(fill_value=0)

print("--- Payment Method Channel Breakdown ---")
display(pay_channel)

--- Payment Method Channel Breakdown ---


Count               Volume              
type           Expense Income       Expense        Income
payment_method                                           
Bank Transfer       81    237 ₹1,064,618.22 ₹5,569,349.20
Cash               400      4   ₹386,837.81    ₹12,592.63
Credit Card       1353      0 ₹3,533,746.50         ₹0.00
Debit Card         320      0   ₹731,408.23         ₹0.00
UPI               2543     62 ₹3,648,406.11 ₹1,175,655.05

In [17]:
# Visualization of Payment Channels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Transaction Count
pay_counts = df.groupby(["payment_method", "type"])["transaction_id"].count().unstack(fill_value=0)
order = pay_counts.sum(axis=1).sort_values(ascending=False).index
pay_counts = pay_counts.reindex(order)

pay_counts.plot(kind="bar", stacked=True, ax=ax1, color={"Expense": "#e74c3c", "Income": "#2ecc71"}, width=0.55)
ax1.set_title("Transaction Count by Payment Method", fontsize=12, fontweight="bold")
ax1.set_ylabel("Count")
ax1.set_xlabel("Payment Method")
ax1.tick_params(axis="x", rotation=25)
ax1.legend(["Expense", "Income"])

# Volume
pay_vols = (df.groupby(["payment_method", "type"])["amount"].sum().unstack(fill_value=0) / 1e5).reindex(order)
pay_vols.plot(kind="bar", stacked=True, ax=ax2, color={"Expense": "#e74c3c", "Income": "#2ecc71"}, width=0.55)
ax2.set_title("Total Monetary Volume by Channel (₹ in Lakhs)", fontsize=12, fontweight="bold")
ax2.set_ylabel("₹ in Lakhs")
ax2.set_xlabel("Payment Method")
ax2.tick_params(axis="x", rotation=25)
ax2.legend(["Expense", "Income"])

plt.tight_layout()
plt.show()

### Payment Method Insights:
- **UPI Dominance in Frequency**: `UPI` represents **2,605 transactions (52.1%)**, handling the vast majority of day-to-day living expenses (groceries, dining, transport).
- **Bank Transfer Dominance in Volume**: While `Bank Transfer` accounts for only 318 transactions (6.4%), it commands **₹66.34 Lakhs (41.1% of total monetary value)** because salary credits, freelance invoices, and rent transfers settle via NEFT/IMPS.
- **Credit Card Role**: `Credit Card` accounts for 1,353 transactions totaling **₹35.34 Lakhs**, heavily utilized for online shopping, utilities, travel bookings, and healthcare.

---
## 10. Monthly and Temporal Analysis

We examine the chronological trajectory across the 12-month observation window (November 2025 – October 2026), evaluating monthly net cash flows, day-of-month patterns, and day-of-week posting behaviors.

> **Caution**: A 12-month window provides a single annual cycle and is **insufficient to confirm multi-year macroeconomic seasonality**. Observed variations reflect the synthetic generation parameters.

In [18]:
# Monthly Aggregation
df["year_month"] = df["date"].dt.to_period("M").astype(str)
monthly_kpis = df.groupby(["year_month", "type"])["amount"].sum().unstack(fill_value=0)
monthly_kpis["Net_Cash_Flow"] = monthly_kpis["Income"] - monthly_kpis["Expense"]
monthly_kpis["Transaction_Count"] = df.groupby("year_month")["transaction_id"].count()

print("--- Monthly Financial Trajectory ---")
display(monthly_kpis)

--- Monthly Financial Trajectory ---


type,Expense,Income,Net_Cash_Flow,Transaction_Count
year_month,,,,
2025-11,"₹826,899.27","₹785,707.48","₹-41,191.79",427
2025-12,"₹1,021,527.45","₹406,361.82","₹-615,165.63",445
2026-01,"₹659,114.87","₹542,363.57","₹-116,751.30",383
2026-02,"₹767,104.24","₹471,713.84","₹-295,390.40",398
2026-03,"₹768,216.07","₹462,408.81","₹-305,807.26",440
2026-04,"₹926,853.97","₹502,851.00","₹-424,002.97",401
2026-05,"₹799,707.78","₹606,029.02","₹-193,678.76",427
2026-06,"₹788,614.47","₹682,118.60","₹-106,495.87",446
2026-07,"₹703,049.48","₹509,741.95","₹-193,307.53",415


In [19]:
# Visualizing Monthly Trajectory
fig, ax = plt.subplots(figsize=(14, 6))
months = monthly_kpis.index
x = np.arange(len(months))
w = 0.35

ax.bar(x - w/2, monthly_kpis["Income"] / 1e5, width=w, label="Income", color="#2ecc71")
ax.bar(x + w/2, monthly_kpis["Expense"] / 1e5, width=w, label="Expense", color="#e74c3c")
ax.plot(x, monthly_kpis["Net_Cash_Flow"] / 1e5, color="#2c3e50", marker="o", linewidth=2.5, label="Net Cash Flow (₹ Lakhs)")

ax.set_title("Monthly Income vs Expense & Net Cash Flow Trajectory", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Amount (₹ in Lakhs)", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(months, rotation=30, ha="right")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
ax.legend(frameon=True, fontsize=10)

plt.tight_layout()
plt.show()

In [20]:
# Day of Month and Day of Week Patterns
df["day_of_month"] = df["date"].dt.day
df["day_name"] = df["date"].dt.day_name()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Day of month
sns.histplot(df["day_of_month"], bins=31, ax=ax1, color="#34495e", discrete=True)
ax1.set_title("Transaction Frequency by Day of Month", fontsize=12, fontweight="bold")
ax1.set_xlabel("Day of Month (1 - 31)")
ax1.set_ylabel("Count")

# Day of week
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_counts = df["day_name"].value_counts().reindex(dow_order)
ax2.bar(dow_counts.index, dow_counts.values, color="#16a085", width=0.55)
ax2.set_title("Transaction Frequency by Day of Week", fontsize=12, fontweight="bold")
ax2.set_ylabel("Count")
ax2.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

### Temporal Insights:
1. **Monthly Expenditure Peaks**: Spending peaks in **December 2025 (₹10.22 Lakhs)** and **April 2026 (₹9.27 Lakhs)**, driven by simulated holiday travel bookings and emergency tech replacements.
2. **Day-of-Month Rhythm**: Transactions are distributed across all 31 days with minor elevations at the start of each month (1st and 5th) due to fixed recurring rent payments and primary salary credits.
3. **Day-of-Week Rhythm**: Activity is distributed fairly evenly across weekdays (690–730 txns/day), with Saturday and Sunday accounting for typical retail and dining outings.

---
## 11. Outlier and Unusual-Transaction Exploration

We distinguish between two distinct concepts:
1. **Statistical Outliers**: Data points falling beyond the statistical upper fence ($Q_3 + 1.5 	imes IQR$) under the empirical distribution of expenses.
2. **Deliberately Injected Synthetic Shocks (`[UNUSUAL]`)**: 12 specific high-magnitude financial events introduced during Phase 2 data generation to serve as ground-truth test cases for the Phase 6 Anomaly Detection module.

> **Crucial Modeling Precaution (Target Leakage)**:
> In the synthetic dataset, unusual transactions carry the prefix `[UNUSUAL]` in the `description` column. During Phase 6 model training, the `description` text **must not be used directly as a feature** to predict anomalies, as doing so would cause severe target leakage. Real-world anomaly detection must identify these shocks from numeric, categorical, and behavioral signals alone.

In [21]:
# Statistical IQR Outlier Detection on Expenses
q1 = exp_df["amount"].quantile(0.25)
q3 = exp_df["amount"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

print(f"Expense Quartile 1 (25%): ₹{q1:,.2f}")
print(f"Expense Quartile 3 (75%): ₹{q3:,.2f}")
print(f"Interquartile Range (IQR): ₹{iqr:,.2f}")
print(f"Statistical Upper Bound (Q3 + 1.5*IQR): ₹{upper_bound:,.2f}")

stat_outliers = exp_df[exp_df["amount"] > upper_bound]
print(f"Total Statistical Outliers: {len(stat_outliers):,} ({len(stat_outliers)/len(exp_df)*100:.2f}% of expenses)")

Expense Quartile 1 (25%): ₹545.20
Expense Quartile 3 (75%): ₹1,910.12
Interquartile Range (IQR): ₹1,364.92
Statistical Upper Bound (Q3 + 1.5*IQR): ₹3,957.50
Total Statistical Outliers: 354 (7.54% of expenses)


In [22]:
# Identifying Deliberate Synthetic Anomaly Examples
unusual_txns = df[df["description"].str.contains(r"\[UNUSUAL\]", regex=True)].copy()

print(f"Total Deliberately Injected [UNUSUAL] Shocks: {len(unusual_txns)}")
display(unusual_txns[["transaction_id", "date", "category", "amount", "payment_method", "description"]])

# Verify overlap between [UNUSUAL] shocks and statistical IQR outliers
unusual_in_iqr = unusual_txns[unusual_txns["amount"] > upper_bound]
print(f"\nDeliberate shocks exceeding statistical IQR upper bound: {len(unusual_in_iqr)} of {len(unusual_txns)} (100%)")

Total Deliberately Injected [UNUSUAL] Shocks: 12

Deliberate shocks exceeding statistical IQR upper bound: 12 of 12 (100%)


,transaction_id,date,category,amount,payment_method,description
123,TXN-00124,2025-11-09,Other Expenses,"₹65,935.31",Debit Card,[UNUSUAL] Emergency Home Renovation & Plumbing...
336,TXN-00337,2025-11-24,Shopping,"₹144,947.07",Credit Card,[UNUSUAL] Apple MacBook Pro M3 Laptop Purchase
434,TXN-00435,2025-12-01,Healthcare,"₹104,782.68",Credit Card,[UNUSUAL] Emergency Hospitalization & Surgery ...
784,TXN-00785,2025-12-25,Shopping,"₹156,809.84",Credit Card,[UNUSUAL] Apple MacBook Pro M3 Laptop Purchase
1432,TXN-01433,2026-02-13,Other Expenses,"₹68,506.23",Debit Card,[UNUSUAL] Emergency Home Renovation & Plumbing...
2080,TXN-02081,2026-03-31,Other Expenses,"₹67,693.52",Debit Card,[UNUSUAL] Emergency Home Renovation & Plumbing...
2341,TXN-02342,2026-04-19,Shopping,"₹161,884.82",Credit Card,[UNUSUAL] Apple MacBook Pro M3 Laptop Purchase
2412,TXN-02413,2026-04-24,Travel,"₹114,751.02",Bank Transfer,[UNUSUAL] International Vacation Flight & Reso...
2891,TXN-02892,2026-05-29,Travel,"₹95,972.54",Bank Transfer,[UNUSUAL] International Vacation Flight & Reso...
2984,TXN-02985,2026-06-05,Shopping,"₹138,257.27",Credit Card,[UNUSUAL] Apple MacBook Pro M3 Laptop Purchase


### Outlier Insights:
- **Statistical Outliers ($\mathbf{N = 354}$)**: Legitimate recurring payments such as monthly rent (₹24,000 – ₹26,000) and substantial family travel bookings (₹10,000 – ₹22,000) naturally exceed the $Q_3 + 1.5 	imes IQR$ upper bound (₹3,957.50). This illustrates why **statistical outliers are not synonymous with fraud or errors** and should never be automatically deleted.
- **Injected Shocks ($\mathbf{N = 12}$)**: All 12 `[UNUSUAL]` transactions represent severe tail risks (₹65,935.31 – ₹161,884.82), spanning emergency surgeries, international travel bookings, gold jewelry purchases, and premium laptop acquisitions. These provide an uncompromised ground-truth benchmark for Phase 6.

---
## 12. Key Findings and Limitations of Synthetic Data

### Key Empirical Findings:
1. **Schema and Data Integrity**: 5,000 transactions, 0 missing values, 0 duplicate IDs, and 100% adherence to the Phase 2 schema contract.
2. **Frequency vs Volume Asymmetry**: Expenses dominate transaction counts (93.94%), but Income contributes 41.91% of monetary volume.
3. **Primary Expenditure Categories**: `Shopping` (₹18.27L) and `Travel` (₹14.65L) lead in monetary magnitude, while `Groceries` (1,412 txns) leads in everyday frequency.
4. **Channel Specialization**: `UPI` accounts for 52.1% of all transaction occurrences, while `Bank Transfer` commands 41.1% of all monetary value.
5. **Skewness & Heavy Tails**: Expense amounts follow a log-normal distribution with median ₹1,021.31, mean ₹1,993.83, and an extreme right tail reaching ₹161,884.82.
6. **Outlier Reality**: 354 transactions exceed the standard IQR upper bound (₹3,957.50, 7.54% of expenses). 12 of these represent deliberately modeled acute financial shocks (`[UNUSUAL]`).

### Limitations of Synthetic Data:
- **Absence of Multi-Year History**: 12 months represents a single annual cycle; long-term trend analysis or multi-year seasonal decomposition cannot be asserted.
- **Deterministic Modeling Rules**: Fixed monthly rent (₹25k) and salary (₹84k) are generated via parameterized stochastic rules rather than organic macroeconomic behavior.
- **Controlled Outliers**: Injected shocks were sampled from pre-specified templates and do not exhibit the full unpredictability of real-world financial fraud or complex multi-account transfers.

---
## 13. Conclusion and Recommendations for Phase 5

The dataset has been thoroughly characterized and verified as structurally sound, statistically clean, and ready for advanced modeling.

### Recommendations for Phase 5 (Feature Engineering):
1. **Log-Transformations**: Apply $\log_{1p}(x)$ transformation to the `amount` variable to stabilize variance and normalize heavy tails for machine learning regressors and clustering algorithms.
2. **Temporal Feature Extraction**: Engineer calendar features including `day_of_week`, `day_of_month`, `is_weekend`, `month`, and `quarter`.
3. **Rolling Window Aggregations**: Compute trailing 7-day and 30-day rolling expense sums, means, and category-level velocity to capture spending acceleration.
4. **Cash Flow Ratios**: Construct daily and monthly expense-to-income ratios and cumulative net balance indicators.
5. **Safeguard Against Target Leakage**: Ensure the `[UNUSUAL]` description prefix is excluded from all feature matrices intended for model training.